In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
# read it to inspect it 
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [ ]:
print('length of dat5aset in characters : ', len(text))

In [ ]:
# lets look at first 1000 characters
print(text[:1000])

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

In [ ]:
# create a mapping from chars to int 
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers 
decode = lambda l: ''.join([itos[i] for i in l ]) # decoder: take a list of integers , output a string
print(encode("hi there"))
print(decode(encode("hii there")))

In [ ]:
# let's encode the entire text dataset and store it into a torch.tensor
import torch #we use Pytorch 
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[:1000])

In [ ]:
# let's now split up the data into train and validation sets 
n = int(0.9*len(data)) # first 90% will be train, rest val 
train_data = data[:n]
val_data = data[n:]

In [ ]:
block_size = 8
train_data[:block_size]

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"when input is {context} the target :{target}")

In [ ]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel7
block_size = 8 # what is the maximum context length for predictions ?

def get_batch(split):
  # generate a small batch of data inputs x and targets y
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix ])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x,y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print('targets :')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
  for t in range(block_size):
    context = xb[b, :t+1]
    target = yb[b,t]
    print(f"when input is {context.tolist()} the target : {target}")

In [ ]:
print(xb)

In [ ]:
import torch
import torch.nn as nn 
from torch.nn import functional as F 
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    # each token directly reads off the logits for the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

  def forward(self, idx, targets=None):
    # idx and targets are both (B,T) tensor of integers 
    logits = self.token_embedding_table(idx) #(B,T,C)
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)
    return logits, loss

  def generate(self, idx, max_new_tokens):
    # idx is (B, T, C) array of indices in the current context 
    for _ in range(max_new_tokens):
      # get the predictions
      logits, loss = self(idx)
      # focus only on the last time step
      logits = logits[:, -1, :] #becomes (B,C)
      # apply softmax to get the probabilities 
      probs = F.softmax(logits, dim=-1) #(B,C)
      # Sample from the distribution
      idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
      # append sampled index to the running sequence
      idx = torch.cat((idx,idx_next),dim=1) #(B, T+1)
    return idx



        

m = BigramLanguageModel(vocab_size)
logits,loss = m(xb, yb)
print(logits.shape)
print(loss)
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long),max_new_tokens=100)[0].tolist()))


In [ ]:
# create a Pytorch optimizer 
optimizer = torch.optim.Adam(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32 
for steps in range(10000):
  # sample a batch of data
  xb, yb = get_batch('train')

  # evaluate the loss 
  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
print(loss.item())

In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long),max_new_tokens=100)[0].tolist()))

In [ ]:
# cosider the following toy example 
torch.manual_seed(1337)
B,T,C = 4,8,2 #batch, time, channels
x = torch.randn(B,T,C)
x.shape

In [ ]:
# we want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1] #t,c
    xbow[b,t] = torch.mean(xprev, 0) 

In [ ]:
# version 2
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1,keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) -----> (B, T, C)
torch.allclose(xbow, xbow2, atol=1e-6)

In [ ]:
# version 3 
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x 
torch.allclose(xbow, xbow3, atol= 1e-6)

In [ ]:
# version 4 : self attention
torch.manual_seed(1337)
B, T, C = 4,8,32 # batch , time , channels
x = torch.randn(B,T,C)
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ x 


In [ ]:
xbow[0], xbow2[0]

In [ ]:
x[0]

In [ ]:
xbow[0]

In [ ]:
torch.tril(torch.ones(3,3))

In [ ]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a,1,keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b 
print('a=')
print(a)
print('---')
print('b=')
print(b)
print('---')
print('c=')
print(c)